# Этап 1. Слои данных, персональные данные и качество

Бронзовый слой лежит в data/bronze, серебряный в таблицах базы и в data/silver, золотой в таблице offers_model и в data/gold.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from db import connect

pd.set_option("display.width", 130, "display.max_columns", 30, "display.max_rows", 60)
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
NAVY, BLUE, YELLOW, GREY = "#17358A", "#2F6FD0", "#F2C230", "#8A94A6"
plt.rcParams.update({"font.family": "Times New Roman", "font.size": 12, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.facecolor": "white", "savefig.dpi": 200,
                     "savefig.bbox": "tight"})


def sql(query, params=None):
    """Запрос к базе PostgreSQL: результат в виде таблицы pandas."""
    with connect() as conn:
        cur = conn.execute(query, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])

In [2]:
# Объём данных в слоях
for path in sorted((ROOT / "data" / "bronze").glob("offers_*.jsonl")):
    with open(path, encoding="utf-8") as f:
        print(path.name, "строк", sum(1 for _ in f), "размер, МБ", round(path.stat().st_size / 1e6, 1))
print(sql("SELECT (SELECT COUNT(*) FROM offers) AS offers, (SELECT COUNT(*) FROM price_history) AS price_history, "
          "(SELECT COUNT(*) FROM v_offers_current) AS current, (SELECT COUNT(*) FROM offers_model) AS gold").to_string(index=False))

offers_2026-10-25_1731.jsonl строк 69 размер, МБ 0.1
offers_2026-10-25_1742.jsonl строк 5972 размер, МБ 12.4
 offers  price_history  current  gold
   5976           6041     5972  5213


In [3]:
# Какие поля остаются в записи бронзового слоя
with open(ROOT / "data" / "bronze" / "offers_2026-10-25_1742.jsonl", encoding="utf-8") as f:
    record = json.loads(f.readline())
print(sorted(record))
print("location:", sorted(record["location"]))
print("point:", record["location"]["point"])
print("offerKey:", record["offerKey"], "buildingKey:", record["buildingKey"])

['agentFee', 'apartment', 'area', 'authorType', 'building', 'buildingKey', 'ceilingHeight', 'creationDate', 'floorsOffered', 'floorsTotal', 'house', 'inMoscow', 'location', 'minRentPeriod', 'offerKey', 'openPlan', 'price', 'rentConditionsMap', 'roomsTotalKey', 'updateDate', 'utilitiesIncluded']
location: ['districts', 'metro', 'metroList', 'point']
point: {'latitude': 55.876, 'longitude': 37.602}
offerKey: 6a404e8a2a6f02a770f0 buildingKey: 5d36fdc6b710be224a0c


In [4]:
# Проверка серебряного и золотого слоёв на персональные данные
from security import PersonalDataGuard

for name in ("silver/offers_silver.csv", "gold/offers_gold.csv"):
    table = pd.read_csv(ROOT / "data" / name, encoding="utf-8-sig", low_memory=False)
    print(name, "столбцов", table.shape[1], "строк", table.shape[0], "находок:", PersonalDataGuard.find_personal_data(table))

silver/offers_silver.csv столбцов 46 строк 5972 находок: []
gold/offers_gold.csv столбцов 19 строк 5213 находок: []


In [5]:
# Журнал проверок качества данных
sql("SELECT source, check_name, ROUND(value::numeric, 4) AS value, threshold, status FROM quality_checks "
    "ORDER BY checked_at, source")

,source,check_name,value,threshold,status
0,offers_2026-10-25_1731.jsonl,"нарушение схемы, доля строк",0.0000,0.02,норма
1,offers_2026-10-25_1731.jsonl,"повторы ключа объявления, строк",0.0000,0.00,норма
2,offers_2026-10-25_1731.jsonl,"пропуски в цене, доля строк",0.0000,0.00,норма
3,offers_2026-10-25_1731.jsonl,"этаж больше этажности, доля строк",0.0000,0.01,норма
4,offers_2026-10-25_1731.jsonl,"персональные данные, находок",0.0000,0.00,норма
5,offers_2026-10-25_1731.jsonl,"возраст данных, дней",10.0000,30.00,норма
6,offers_2026-10-25_1742.jsonl,"нарушение схемы, доля строк",0.0000,0.02,норма
7,offers_2026-10-25_1742.jsonl,"повторы ключа объявления, строк",0.0000,0.00,норма
8,offers_2026-10-25_1742.jsonl,"пропуски в цене, доля строк",0.0000,0.00,норма
9,offers_2026-10-25_1742.jsonl,"этаж больше этажности, доля строк",0.0005,0.01,норма


In [6]:
# Механизм обновления: два сбора, журнал загрузок и история цен
print(sql("SELECT run_id, file_name, offers_count, new_offers, price_changes FROM load_runs ORDER BY run_id").to_string(index=False))
print(sql("SELECT COUNT(*) FILTER (WHERE is_active) AS active, COUNT(*) FILTER (WHERE NOT is_active) AS inactive FROM offers").to_string(index=False))
print(sql("SELECT run_id, price FROM price_history WHERE offer_key IN (SELECT offer_key FROM price_history "
          "GROUP BY offer_key HAVING COUNT(DISTINCT price) > 1) ORDER BY run_id").to_string(index=False))

 run_id                    file_name  offers_count  new_offers  price_changes
      1 offers_2026-10-25_1731.jsonl            69          69              0
      2 offers_2026-10-25_1742.jsonl          5972        5907              1


 active  inactive
   5972         4


 run_id  price
      1  80000
      2  89500


In [7]:
# Разграничение доступа: права роли rent_reader
sql("SELECT table_name, privilege_type FROM information_schema.role_table_grants WHERE grantee = 'rent_reader' ORDER BY table_name")

,table_name,privilege_type
0,load_runs,SELECT
1,metro_stations,SELECT
2,model_runs,SELECT
3,offers,SELECT
4,offers_model,SELECT
5,pipeline_runs,SELECT
6,price_history,SELECT
7,quality_checks,SELECT
8,v_offers_current,SELECT
